# Graph Gallery by Dataset, Method, and Size

This notebook:
- scans generated graph pickles from `samples/drive_graphs_snapshot_6may/graphs`
- loads reference graphs from the size-reference dataset roots
- plots representative graphs by method and graph size for SBM, tree, and PA
- uses the PA degree-circle layout logic from `plot_graph_grid` for PA graphs


In [ ]:
from __future__ import annotations

import math
import pickle
import re
import shutil
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx

plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"]
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["axes.unicode_minus"] = False

if shutil.which("latex"):
    plt.rcParams["text.usetex"] = True


In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists():
    REPO_ROOT = Path("/mnt/lts4/scratch/home/mmonteir/jacobi_diff/jacobi-graph-diffusion")

GENERATED_ROOT = REPO_ROOT / "samples" / "drive_graphs_snapshot_6may" / "graphs"
FIGURE_OUT_DIR = REPO_ROOT / "analysis" / "figures" / "method_size_gallery"

METHOD_ORDER = ["dataset", "digress", "defog", "gdss", "grum", "diphon"]
METHOD_LABELS = {
    "dataset": "Dataset",
    "defog": "DeFoG",
    "digress": "DiGress",
    "diphon": "DiPhon",
    "gdss": "GDSS",
    "grum": "GruM",
}

DATASET_CONFIGS = {
    "sbm": {
        "label": "SBM",
        "dataset_name": "sbm_2comms_graphon",
        "dataset_root": REPO_ROOT / "data" / "size_ref" / "sbm_2comms_graphon",
        "full_dataset": REPO_ROOT / "data" / "size_ref" / "sbm_2comms_graphon" / "sbm_2comms_graphon.pkl",
        "sizes": [40, 60, 80, 100, 120, 140, 160, 180, 200],
    },
    "tree": {
        "label": "Tree",
        "dataset_name": "tree_graphon",
        "dataset_root": REPO_ROOT / "data" / "size_ref" / "tree_graphon",
        "full_dataset": REPO_ROOT / "data" / "size_ref" / "tree_graphon" / "tree_graphon.pkl",
        "sizes": [20, 40, 60, 80, 100, 120, 140, 160, 180, 200],
    },
    "pa": {
        "label": "PA",
        "dataset_name": "pa_graphon",
        "dataset_root": REPO_ROOT / "data" / "size_ref" / "pa_graphon",
        "full_dataset": REPO_ROOT / "data" / "size_ref" / "pa_graphon" / "pa_graphon.pkl",
        "sizes": [40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300],
    },
}

DEFAULT_DATASET = "sbm"
DEFAULT_SIZES = DATASET_CONFIGS[DEFAULT_DATASET]["sizes"]

FIGURE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("GENERATED_ROOT:", GENERATED_ROOT)
print("FIGURE_OUT_DIR:", FIGURE_OUT_DIR)


In [ ]:
def degree_circle_layout(graph: nx.Graph) -> dict:
    degrees = dict(graph.degree())
    if not degrees:
        return nx.circular_layout(graph)

    nodes_sorted = sorted(degrees, key=degrees.get, reverse=True)
    n_nodes = len(nodes_sorted)
    angles = [2.0 * 3.141592653589793 * i / n_nodes for i in range(n_nodes)]
    return {
        node: (math.cos(theta), math.sin(theta))
        for node, theta in zip(nodes_sorted, angles)
    }


def compute_layout(graph: nx.Graph, dataset_key: str, seed: int) -> dict:
    if dataset_key == "pa":
        return degree_circle_layout(graph)
    return nx.spring_layout(graph, seed=seed)


def parse_size_from_name(path: Path) -> int | None:
    name = path.name
    patterns = [
        r"(?:^|_)n(\d+)(?:_|\.pkl$)",
        r"(?:^|_)(\d+)(?:_(?:gdss|grum)|\.pkl$)",
    ]
    for pattern in patterns:
        m = re.search(pattern, name)
        if m:
            return int(m.group(1))
    return None


def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def extract_graph_list(obj, split_preference=("test", "val", "train")):
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for split in split_preference:
            if split in obj and isinstance(obj[split], list):
                return obj[split]
        for value in obj.values():
            if isinstance(value, list):
                return value
    raise TypeError(f"Unsupported pickle payload type: {type(obj).__name__}")


def load_graphs(path: Path):
    return extract_graph_list(load_pickle(path))


def dataset_config(dataset_key: str):
    if dataset_key not in DATASET_CONFIGS:
        raise KeyError(f"Unknown dataset_key={dataset_key!r}; choose from {sorted(DATASET_CONFIGS)}")
    return DATASET_CONFIGS[dataset_key]


def discover_generated_pickles(dataset_key: str):
    root = GENERATED_ROOT / dataset_key
    index = defaultdict(dict)
    if not root.exists():
        return index
    for method_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        method = method_dir.name
        for path in sorted(method_dir.glob("*.pkl")):
            size = parse_size_from_name(path)
            if size is None:
                continue
            index[method][size] = path
    return index


def load_dataset_graphs_by_size(dataset_key: str):
    cfg = dataset_config(dataset_key)
    all_train_graphs = extract_graph_list(load_pickle(cfg["full_dataset"]), split_preference=("train",))
    graphs_by_size = defaultdict(list)
    for graph in all_train_graphs:
        graphs_by_size[graph.number_of_nodes()].append(graph)
    return graphs_by_size


def graph_for_index(graphs, index: int):
    if not graphs:
        return None
    return graphs[index % len(graphs)]


In [ ]:
def draw_graph(ax, graph, *, dataset_key: str, seed: int = 7, missing_label: str = ""):
    cfg = dataset_config(dataset_key)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    if graph is None:
        if missing_label:
            ax.text(0.5, 0.5, missing_label, ha="center", va="center", fontsize=13)
        return

    if graph.number_of_nodes() == 0:
        ax.text(0.5, 0.5, "empty", ha="center", va="center", fontsize=13)
        return

    pos = compute_layout(graph, dataset_key, seed)
    if dataset_key == "pa":
        degrees = dict(graph.degree())
        max_degree = max(degrees.values()) if degrees else 0
        node_color = [degrees[node] / max_degree for node in graph.nodes()] if max_degree > 0 else "#0f766e"
        nx.draw_networkx_edges(graph, pos, ax=ax, edge_color="#a8b3bd", width=0.35, alpha=0.7)
        nx.draw_networkx_nodes(
            graph,
            pos,
            ax=ax,
            node_color=node_color,
            cmap="viridis" if max_degree > 0 else None,
            node_size=max(18, 1800 / max(graph.number_of_nodes(), 1)),
            linewidths=0,
            alpha=0.95,
        )
        return

    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color="#9aa5b1", width=0.8, alpha=0.85)
    nx.draw_networkx_nodes(
        graph,
        pos,
        ax=ax,
        node_color="#0f766e",
        node_size=max(12, 2400 / max(graph.number_of_nodes(), 1)),
        linewidths=0,
        alpha=0.95,
    )


def render_method_size_gallery(
    dataset_key: str = DEFAULT_DATASET,
    sizes=None,
    sample_index: int = 0,
    methods=None,
    figsize_scale: float = 2.9,
    header_fontsize: int = 16,
    row_fontsize: int = 16,
):
    cfg = dataset_config(dataset_key)
    sizes = list(sizes or cfg["sizes"])
    methods = list(methods or METHOD_ORDER)
    generated_index = discover_generated_pickles(dataset_key)
    dataset_graphs_by_size = load_dataset_graphs_by_size(dataset_key)

    n_rows = len(methods)
    n_cols = len(sizes)
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_scale * n_cols, figsize_scale * n_rows),
        squeeze=False,
    )

    for col, size in enumerate(sizes):
        axes[0, col].set_title(rf"$N = {size}$", fontsize=header_fontsize, pad=12)

        dataset_graph = graph_for_index(dataset_graphs_by_size.get(size, []), sample_index)
        draw_graph(axes[0, col], dataset_graph, dataset_key=dataset_key, seed=size, missing_label="")

        for row, method in enumerate(methods[1:], start=1):
            path = generated_index.get(method, {}).get(size)
            graph = None
            if path is not None:
                graph = graph_for_index(load_graphs(path), sample_index)
            draw_graph(axes[row, col], graph, dataset_key=dataset_key, seed=size)

    for row, method in enumerate(methods):
        axes[row, 0].annotate(
            METHOD_LABELS.get(method, method),
            xy=(-0.14, 0.5),
            xycoords="axes fraction",
            ha="right",
            va="center",
            fontsize=row_fontsize,
            fontweight="bold",
            rotation=90,
        )

    fig.tight_layout(w_pad=0.25, h_pad=0.5)
    return fig, axes


In [ ]:
def summarize_available_graphs(dataset_key: str):
    generated_index = discover_generated_pickles(dataset_key)
    available_sizes = sorted({size for method_sizes in generated_index.values() for size in method_sizes})
    dataset_sizes = sorted(load_dataset_graphs_by_size(dataset_key))

    print(dataset_config(dataset_key)["label"])
    print("Available generated sizes:", available_sizes)
    print("Dataset train sizes:", dataset_sizes)
    for method in METHOD_ORDER[1:]:
        print(f"{METHOD_LABELS.get(method, method):>8}:", sorted(generated_index.get(method, {})))
    print()


for dataset_key in DATASET_CONFIGS:
    summarize_available_graphs(dataset_key)


## SBM


In [ ]:
SAMPLE_INDEX = 0

fig, axes = render_method_size_gallery(dataset_key="sbm", sample_index=SAMPLE_INDEX)
plt.show()


## Tree


In [ ]:
fig, axes = render_method_size_gallery(dataset_key="tree", sample_index=SAMPLE_INDEX)
plt.show()


## PA


In [ ]:
fig, axes = render_method_size_gallery(dataset_key="pa", sample_index=SAMPLE_INDEX, figsize_scale=2.4)
plt.show()


## PA Method Grids


In [ ]:
def render_pa_method_grids(sizes=None, sample_index: int = 0, methods=None):
    dataset_key = "pa"
    cfg = dataset_config(dataset_key)
    sizes = list(sizes or cfg["sizes"])
    methods = list(methods or METHOD_ORDER)
    generated_index = discover_generated_pickles(dataset_key)
    dataset_graphs_by_size = load_dataset_graphs_by_size(dataset_key)

    figures = {}
    for method in methods:
        graphs = []
        used_sizes = []
        for size in sizes:
            if method == "dataset":
                graph = graph_for_index(dataset_graphs_by_size.get(size, []), sample_index)
            else:
                path = generated_index.get(method, {}).get(size)
                graph = graph_for_index(load_graphs(path), sample_index) if path is not None else None
            if graph is not None:
                graphs.append(graph)
                used_sizes.append(size)

        if not graphs:
            continue

        fig, axes = plt.subplots(1, len(graphs), figsize=(3 * len(graphs), 3), squeeze=False)
        axes = axes.ravel()
        for ax, graph, size in zip(axes, graphs, used_sizes):
            draw_graph(ax, graph, dataset_key=dataset_key, seed=size)
            ax.set_title(rf"$N = {size}$", fontsize=14, pad=8)
        fig.suptitle(METHOD_LABELS.get(method, method), fontsize=16)
        fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.92])
        figures[method] = fig
    return figures


pa_grid_figures = render_pa_method_grids(sample_index=SAMPLE_INDEX)
for fig in pa_grid_figures.values():
    plt.show()


## Save Figures


In [ ]:
def save_gallery(dataset_key: str, sample_index: int = 0):
    cfg = dataset_config(dataset_key)
    sizes = cfg["sizes"]
    out_dir = FIGURE_OUT_DIR / dataset_key
    out_dir.mkdir(parents=True, exist_ok=True)
    tag = "_".join(map(str, sizes))

    fig, axes = render_method_size_gallery(
        dataset_key=dataset_key,
        sizes=sizes,
        sample_index=sample_index,
        figsize_scale=2.4 if dataset_key == "pa" else 2.9,
    )
    png_path = out_dir / f"{dataset_key}_gallery_sizes_{tag}_sample_{sample_index}.png"
    pdf_path = out_dir / f"{dataset_key}_gallery_sizes_{tag}_sample_{sample_index}.pdf"
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    return png_path, pdf_path


def save_pa_method_grid_figures(sample_index: int = 0):
    out_dir = FIGURE_OUT_DIR / "pa" / "method_grids"
    out_dir.mkdir(parents=True, exist_ok=True)
    figures = render_pa_method_grids(sample_index=sample_index)
    saved = []
    for method, fig in figures.items():
        png_path = out_dir / f"pa_{method}_method_grid_sample_{sample_index}.png"
        pdf_path = out_dir / f"pa_{method}_method_grid_sample_{sample_index}.pdf"
        fig.savefig(png_path, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
        plt.close(fig)
        saved.append((png_path, pdf_path))
    return saved


saved = []
for dataset_key in ["sbm", "tree", "pa"]:
    saved.append(save_gallery(dataset_key, sample_index=SAMPLE_INDEX))
saved.extend(save_pa_method_grid_figures(sample_index=SAMPLE_INDEX))

print("Saved:")
for paths in saved:
    for path in paths:
        print(path)
